**1. Install dan Import libraries**
Cell ini adalah fondasi dari seluruh notebook. Pertama, menginstall semua library yang dibutuhkan via pip. Kemudian mengimport semua modul yang akan dipakai di cell-cell berikutnya, mulai dari operasi file, manipulasi data, hingga PyTorch untuk deep learning.

In [1]:
!pip install -q tqdm scipy pandas numpy torch

import os, json, random, datetime, gzip, shutil, zipfile
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import tqdm  # ← langsung import class-nya
import torch
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from scipy.sparse import csr_matrix

np.random.seed(12345)
random.seed(12345)

OUTPUT_DIR = './processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Setup selesai.")


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Setup selesai.


**2. Download Dataset**
Mengunduh raw dataset langsung dari sumbernya ke dalam Colab, lalu menyimpannya di folder ./raw/. Tersedia tiga dataset sekaligus dengan toggle True/False.

**JIKA LINK DOWNLOAD GAGAL, GUNAKAN LINK https://jmcauley.ucsd.edu/data/amazon_v2/categoryFilesSmall/Beauty_5.json.gz**

* Sumber download:
1. Beauty — dari datarepo.eng.ucsd.edu (McAuley Lab UCSD), versi Amazon Reviews 2018, subset Beauty 5-core (~198K reviews). File .json.gz diunduh lalu diekstrak menjadi .json.
2. ML-1M — dari GroupLens (files.grouplens.org), berisi 1 juta rating film dari ~6.000 user.
3. LastFM — dari HetRec 2011 (files.grouplens.org), berisi data tag artis dengan timestamp.

In [2]:
import os, gzip, shutil, requests, zipfile
from tqdm import tqdm

DOWNLOAD_BEAUTY  = True
DOWNLOAD_ML1M    = True
DOWNLOAD_LASTFM  = True


def download_file(url, dest_path, desc='Downloading'):
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    response = requests.get(url, stream=True, timeout=60)
    if response.status_code != 200:
        print(f"❌ Gagal download (HTTP {response.status_code}): {url}")
        return False
    total = int(response.headers.get('content-length', 0))
    with open(dest_path, 'wb') as f, tqdm(
        desc=desc, total=total, unit='B', unit_scale=True, unit_divisor=1024
    ) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            bar.update(len(chunk))
    return True


# --- Beauty ---
if DOWNLOAD_BEAUTY:
    RAW_DIR   = './raw/Beauty'
    gz_path   = f'{RAW_DIR}/Beauty_5.json.gz'
    json_path = f'{RAW_DIR}/Beauty.json'

    if os.path.exists(json_path) and os.path.getsize(json_path) > 0:
        print(f"✅ Beauty sudah ada, skip download ({os.path.getsize(json_path)/1024/1024:.1f} MB)")
    else:
        os.makedirs(RAW_DIR, exist_ok=True)
        URL = 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Beauty_5.json.gz'
        print("Downloading Beauty...")
        success = download_file(URL, gz_path, desc='Beauty')
        if success:
            print("Extracting...")
            with gzip.open(gz_path, 'rb') as f_in:
                with open(json_path, 'wb') as f_out:
                    shutil.copyfileobj(f_in, f_out)
            os.remove(gz_path)  # hapus .gz setelah ekstrak untuk hemat storage
            print(f"✅ Beauty siap di {json_path}")
        else:
            print("❌ Download gagal.")


# --- ML-1M ---
if DOWNLOAD_ML1M:
    RAW_DIR  = './raw/ml-1m'
    dat_path = f'{RAW_DIR}/ratings.dat'
    zip_path = './raw/ml1m.zip'

    if os.path.exists(dat_path) and os.path.getsize(dat_path) > 0:
        print(f"✅ ML-1M sudah ada, skip download ({os.path.getsize(dat_path)/1024/1024:.1f} MB)")
    else:
        os.makedirs(RAW_DIR, exist_ok=True)
        print("\nDownloading ML-1M...")
        success = download_file(
            'https://files.grouplens.org/datasets/movielens/ml-1m.zip',
            zip_path, desc='ML-1M'
        )
        if success:
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extract('ml-1m/ratings.dat', './raw/')
            os.remove(zip_path)
            print(f"✅ ML-1M siap di {dat_path}")


# --- LastFM ---
if DOWNLOAD_LASTFM:
    RAW_DIR  = './raw/LastFM'
    dat_path = f'{RAW_DIR}/user_taggedartists-timestamps.dat'
    zip_path = './raw/lastfm.zip'

    if os.path.exists(dat_path) and os.path.getsize(dat_path) > 0:
        print(f"✅ LastFM sudah ada, skip download ({os.path.getsize(dat_path)/1024/1024:.1f} MB)")
    else:
        os.makedirs(RAW_DIR, exist_ok=True)
        print("\nDownloading LastFM...")
        success = download_file(
            'https://files.grouplens.org/datasets/hetrec-2011/hetrec2011-lastfm-2k.zip',
            zip_path, desc='LastFM'
        )
        if success:
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(RAW_DIR)
            os.remove(zip_path)
            print(f"✅ LastFM siap di {RAW_DIR}/")

✅ Beauty sudah ada, skip download (134.3 MB)
✅ ML-1M sudah ada, skip download (23.5 MB)
✅ LastFM sudah ada, skip download (5.0 MB)


**3. Utility Functions**
Mendefinisikan empat fungsi inti yang menjadi tulang punggung preprocessing. Fungsi-fungsi ini tidak langsung dijalankan di sini, melainkan dipanggil oleh process() di Cell 5.

* Penjelasan per Fungsi
1. get_interaction(datas)
Menerima list of tuples (user, item, timestamp), lalu mengelompokkan interaksi per user dan mengurutkannya secara kronologis. Juga menghitung time interval, yaitu selisih waktu (dalam detik) antara dua interaksi berurutan. Interaksi pertama diberi interval 0.

2. check_Kcore(user_items, user_core, item_core)
Menghitung frekuensi kemunculan setiap user dan item. Ini dipakai untuk mengecek apakah semua user dan item sudah memenuhi syarat minimum K interaksi.

3. filter_Kcore(user_items, time_interval, user_core, item_core)
Melakukan K-core filtering secara iteratif. Artinya, user yang interaksinya kurang dari user_core dan item yang interaksinya kurang dari item_core dihapus, lalu dicek ulang sampai tidak ada perubahan lagi (konvergen). BSARec memakai standar 5-core, artinya setiap user dan item minimal harus punya 5 interaksi.

4. id_map(user_items, time_interval)
Meremap user ID dan item ID asli (yang bisa berupa string acak seperti A2SUAM1J3GNN3B) menjadi integer berurutan mulai dari 1. Ini diperlukan karena model butuh ID numerik untuk embedding lookup.

In [3]:
def get_interaction(datas):
    user_time = defaultdict(list)
    for user, item, time in datas:
        user_time[user].append((item, time))

    user_items = {}
    time_interval = {}

    for user, records in tqdm(user_time.items(), desc='Sorting'):
        records.sort(key=lambda x: x[1])
        items = [r[0] for r in records]
        times = [r[1] for r in records]
        intervals = [0] + [times[i] - times[i-1] for i in range(1, len(times))]
        user_items[user] = items
        time_interval[user] = intervals

    return user_items, time_interval


def check_Kcore(user_items, user_core=5, item_core=5):
    user_count = defaultdict(int)
    item_count = defaultdict(int)

    for user, items in user_items.items():
        for item in items:
            user_count[user] += 1
            item_count[item] += 1

    return user_count, item_count

def filter_Kcore(user_items, time_interval, user_core=5, item_core=5):
    while True:
        user_count, item_count = check_Kcore(
            user_items, user_core, item_core
        )
        valid_users = {
            user for user, count in user_count.items()
            if count >= user_core
        }
        valid_items = {
            item for item, count in item_count.items()
            if count >= item_core
        }

        new_user_items = {}
        new_time_interval = {}

        for user, items in user_items.items():
            if user not in valid_users:
                continue

            filtered = [
                (item, interval)
                for item, interval in zip(items, time_interval[user])
                if item in valid_items
            ]
            if not filtered:
                continue

            new_items, new_intervals = zip(*filtered)
            new_user_items[user] = list(new_items)
            new_time_interval[user] = list(new_intervals)

        if new_user_items == user_items:
            break

        user_items = new_user_items
        time_interval = new_time_interval

    return user_items, time_interval


def id_map(user_items, time_interval):
    all_items = set()
    for items in user_items.values():
        all_items.update(items)

    item2id = {item: idx + 1 for idx, item in enumerate(sorted(all_items))}
    user2id = {user: idx + 1 for idx, user in enumerate(sorted(user_items.keys()))}

    new_user_items = {}
    new_time_interval = {}

    for user, items in user_items.items():
        uid = str(user2id[user])
        new_user_items[uid] = [str(item2id[i]) for i in items]
        new_time_interval[uid] = time_interval[user]

    data_maps = {'user2id': user2id, 'item2id': item2id}
    return new_user_items, new_time_interval, len(user2id), len(item2id), data_maps


print("Utility functions siap.")

Utility functions siap.


**4. Dataset Loaders**
Mendefinisikan fungsi-fungsi loader yang membaca raw file masing-masing dataset ke dalam format standar: list of tuples (user, item, timestamp). Format ini yang kemudian diterima oleh get_interaction() di Cell 3.

* Penjelasan per loader:

1. load_amazon(dataset_name, rating_score)
Membaca file .json Amazon baris per baris. Setiap baris adalah satu review. Field yang diambil: reviewerID (user), asin (item ID produk), unixReviewTime (timestamp). Review dengan rating ≤ rating_score dibuang — defaultnya 0.0 berarti semua rating dipakai.

2. load_ml1m()
Membaca file ratings.dat yang formatnya khusus dengan delimiter ::. Diparse menggunakan pandas. Field yang diambil: user, item, timestamp.

3. load_lastfm()
Membaca file user_taggedartists-timestamps.dat. Baris pertama adalah header sehingga di-skip. Timestamp aslinya dalam milidetik, dibagi 1000 untuk dikonversi ke detik agar konsisten dengan dataset lain.

In [4]:
def load_amazon(dataset_name, rating_score=0.0):
    data_file = f'./raw/{dataset_name}/{dataset_name}.json'
    if not os.path.exists(data_file):
        raise FileNotFoundError(
            f"File tidak ditemukan: {data_file}\n"
            f"Pastikan Cell 2 sudah dijalankan dan download berhasil."
        )
    print(f"Membaca: {data_file}")
    datas = []
    with open(data_file, encoding='utf-8') as f:
        for line in tqdm(f, desc=f'Loading {dataset_name}'):
            line = line.strip()
            if not line:
                continue
            try:
                inter = json.loads(line)
            except json.JSONDecodeError:
                continue
            if float(inter.get('overall', 0)) <= rating_score:
                continue
            user = inter.get('reviewerID')
            item = inter.get('asin')
            time = inter.get('unixReviewTime')
            if user and item and time:
                datas.append((user, item, int(time)))
    return datas


def load_ml1m():
    filepath = './raw/ml-1m/ratings.dat'
    df = pd.read_csv(
        filepath, delimiter='::', header=None, engine='python',
        names=['user', 'item', 'rating', 'timestamp']
    )[['user', 'item', 'timestamp']]
    datas = [
        (str(row['user']), str(row['item']), int(row['timestamp']))
        for _, row in tqdm(df.iterrows(), total=len(df), desc='Loading ML-1M')
    ]
    return datas


def load_lastfm():
    data_file = './raw/LastFM/user_taggedartists-timestamps.dat'
    datas = []
    with open(data_file) as f:
        lines = f.readlines()[1:]
    for line in tqdm(lines, desc='Loading LastFM'):
        user, item, _, timestamp = line.strip().split('\t')
        datas.append((user, item, int(timestamp) // 1000))
    return datas


print("Loaders siap.")

Loaders siap.


**5. Fungsi Proses Utama**
Mendefinisikan fungsi process() yang mengorkestrasikan seluruh pipeline preprocessing dari awal hingga akhir dalam satu pemanggilan fungsi.

* Urutan pipeline di dalam process():
1. Load — memanggil loader yang sesuai berdasarkan nama dataset
2. Sort & time intervals — memanggil get_interaction()
3. K-core filtering — memanggil filter_Kcore() dengan user_core=5, item_core=5
4. Remap ID — memanggil id_map()
5. Cetak statistik — menampilkan jumlah user, item, interaksi, rata-rata panjang sequence, dan sparsity
6. Simpan — menulis dua file output ke OUTPUT_DIR

In [5]:
def process(data_name, user_core=5, item_core=5):
    print(f"\n{'='*50}\n Processing: {data_name}\n{'='*50}")

    item_file = os.path.join(OUTPUT_DIR, f'{data_name}.txt')
    time_file = os.path.join(OUTPUT_DIR, f'{data_name}_time.txt')
    if all(
        os.path.isfile(path) and os.path.getsize(path) > 0
        for path in (item_file, time_file)
    ):
        print(f"Output {data_name} sudah ada dan tidak kosong; skip pemrosesan.")
        return

    if data_name in ['Beauty', 'Sports_and_Outdoors', 'Toys_and_Games']:
        datas = load_amazon(data_name)
    elif data_name == 'ML-1M':
        datas = load_ml1m()
    elif data_name == 'LastFM':
        datas = load_lastfm()
    else:
        raise ValueError(f"Dataset tidak dikenal: {data_name}")

    print("Jumlah data:", len(datas))
    print("Contoh data:", datas[:3])

    user_items, time_interval = get_interaction(datas)
    user_count, item_count = check_Kcore(user_items)
    print("Sebelum filter — users:", len(user_count), "items:", len(item_count))
    print("Interaksi minimum per user:", min(user_count.values(), default=0))
    print("Interaksi minimum per item:", min(item_count.values(), default=0))


    user_items, time_interval = filter_Kcore(user_items, time_interval, user_core, item_core)
    print(f"Setelah {user_core}-core: {len(user_items):,} users")

    user_items, time_interval, num_users, num_items, data_maps = id_map(user_items, time_interval)
    if num_users == 0 or num_items == 0:
        raise ValueError(
            f"Tidak ada interaksi tersisa untuk {data_name} setelah filter "
            f"{user_core}-core/{item_core}-core. Periksa data dan jalankan ulang "
            "cell fungsi dari awal."
        )

    seq_lengths = [len(seq) for seq in user_items.values()]
    interact_num = sum(seq_lengths)
    sparsity = (1 - interact_num / (num_users * num_items)) * 100

    print(f"\nUsers        : {num_users:,}")
    print(f"Items        : {num_items:,}")
    print(f"Interactions : {interact_num:,}")
    print(f"Avg seq len  : {np.mean(seq_lengths):.2f}")
    print(f"Sparsity     : {sparsity:.2f}%")

    with open(item_file, 'w') as f_item, open(time_file, 'w') as f_time:
        for user, items in user_items.items():
            f_item.write(user + ' ' + ' '.join(items) + '\n')
            f_time.write(user + ' ' + ' '.join(map(str, time_interval[user])) + '\n')

    print(f"\nTersimpan: {item_file}")
    print(f"Tersimpan: {time_file}")


print("process() siap.")

process() siap.


**6. Jalankan Pre-Processing**
Menjalankan cell utama. Memanggil process() untuk setiap dataset yang sudah di download di cell 2

In [6]:
# Komentari dataset yang belum didownload
process('Beauty')
process('ML-1M')
process('LastFM')


 Processing: Beauty
Output Beauty sudah ada dan tidak kosong; skip pemrosesan.

 Processing: ML-1M
Output ML-1M sudah ada dan tidak kosong; skip pemrosesan.

 Processing: LastFM
Output LastFM sudah ada dan tidak kosong; skip pemrosesan.


**7. RecDataSet & DataLoader**
Mendefinisikan class RecDataset (PyTorch Dataset) dan beberapa fungsi helper untuk membaca hasil preprocessing dan menyiapkannya untuk training model BSARec.

* Penjelasan komponen:
1. neg_sample(item_set, item_size)
2. Mengambil satu item negatif secara acak, yaitu item yang belum pernah diinteraksikan oleh user tersebut. Dipakai saat training sebagai contoh negatif.

* RecDataset
Class Dataset PyTorch dengan tiga mode:
1. train — membuat semua subsequence dari tiap sequence user. Misalnya sequence [A, B, C, D] menghasilkan training sample [A]→B, [A,B]→C, [A,B,C]→D.Semua di-pad dengan 0 di kiri hingga panjang max_seq_length.
2. valid — menggunakan semua item kecuali item terakhir sebagai input, item terakhir sebagai target. Item terakhir untuk validasi, item kedua dari belakang untuk test.
3. test — menggunakan seluruh sequence sebagai input.

* get_user_seqs(data_file)
Membaca file .txt hasil Cell 6 dan mengembalikan list of sequences, item maksimum, dan jumlah user.

* get_dataloader(args, user_seq)
Membungkus RecDataset ke dalam PyTorch DataLoader untuk train, valid, dan test. Train memakai RandomSampler (diacak), valid dan test memakai SequentialSampler (berurutan).

In [7]:
def neg_sample(item_set, item_size):
    item = random.randint(1, item_size - 1)
    while item in item_set:
        item = random.randint(1, item_size - 1)
    return item


class RecDataset(Dataset):
    def __init__(self, args, user_seq, data_type='train'):
        self.args = args
        self.max_len = args.max_seq_length
        self.data_type = data_type
        self.contrastive_learning = args.model_type.lower() in ['fearec', 'duorec']
        self.user_seq = []
        self.user_ids = []

        if data_type == 'train':
            for user, seq in enumerate(user_seq):
                input_ids = seq[-(self.max_len + 2):-2]
                for i in range(len(input_ids)):
                    self.user_seq.append(input_ids[:i + 1])
                    self.user_ids.append(user)
        elif data_type == 'valid':
            for sequence in user_seq:
                self.user_seq.append(sequence[:-1])
        else:
            self.user_seq = user_seq

    def __len__(self):
        return len(self.user_seq)

    def __getitem__(self, index):
        items = self.user_seq[index]
        input_ids = items[:-1]
        answer = items[-1]

        seq_set = set(items)
        neg_answer = neg_sample(seq_set, self.args.item_size)

        pad_len = self.max_len - len(input_ids)
        input_ids = [0] * pad_len + input_ids
        input_ids = input_ids[-self.max_len:]

        if self.data_type in ['valid', 'test']:
            return (
                torch.tensor(index, dtype=torch.long),
                torch.tensor(input_ids, dtype=torch.long),
                torch.tensor(answer, dtype=torch.long),
                torch.zeros(0, dtype=torch.long),
                torch.zeros(0, dtype=torch.long),
            )
        else:
            return (
                torch.tensor(self.user_ids[index], dtype=torch.long),
                torch.tensor(input_ids, dtype=torch.long),
                torch.tensor(answer, dtype=torch.long),
                torch.tensor(neg_answer, dtype=torch.long),
                torch.zeros(0, dtype=torch.long),
            )


def get_user_seqs(data_file):
    user_seq, item_set = [], set()
    with open(data_file) as f:
        for line in f:
            _, *items = line.strip().split()
            items = [int(i) for i in items]
            user_seq.append(items)
            item_set.update(items)
    return user_seq, max(item_set), len(user_seq)


def get_dataloader(args, user_seq):
    train_ds = RecDataset(args, user_seq, data_type='train')
    valid_ds = RecDataset(args, user_seq, data_type='valid')
    test_ds  = RecDataset(args, user_seq, data_type='test')

    train_loader = DataLoader(train_ds, sampler=RandomSampler(train_ds),
                              batch_size=args.batch_size, num_workers=args.num_workers)
    valid_loader = DataLoader(valid_ds, sampler=SequentialSampler(valid_ds),
                              batch_size=args.batch_size, num_workers=args.num_workers)
    test_loader  = DataLoader(test_ds,  sampler=SequentialSampler(test_ds),
                              batch_size=args.batch_size, num_workers=args.num_workers)
    return train_loader, valid_loader, test_loader


print("RecDataset & DataLoader siap.")

RecDataset & DataLoader siap.


**8. Demo DataLoader**
Memverifikasi bahwa seluruh pipeline dari Cell 1–7 berjalan benar dengan mencoba load data dan mengambil satu batch.

Cara menggunakannya:
Ganti variabel DATASET sesuai dataset yang sudah diproses:

In [ ]:
from types import SimpleNamespace

DATASET = 'ML-1M'  # ganti sesuai kebutuhan: 'ML-1M' atau 'LastFM'
data_file = os.path.join(OUTPUT_DIR, f'{DATASET}.txt')

user_seq, max_item, num_users = get_user_seqs(data_file)

args = SimpleNamespace(
    model_type     = 'bsarec',
    max_seq_length = 50,
    item_size      = max_item + 1,
    batch_size     = 256,
    num_workers    = 2,
)

train_loader, valid_loader, test_loader = get_dataloader(args, user_seq)

print(f"Dataset       : {DATASET}")
print(f"Users         : {num_users:,}")
print(f"Items         : {max_item + 1:,}")
print(f"Train batches : {len(train_loader):,}")
print(f"Valid batches : {len(valid_loader):,}")
print(f"Test batches  : {len(test_loader):,}")

uid, inp, ans, neg, _ = next(iter(train_loader))
print(f"\ninput_ids shape : {inp.shape}")
print(f"answers shape   : {ans.shape}")

Dataset       : ML-1M
Users         : 6,040
Items         : 3,417
Train batches : 1,047
Valid batches : 24
Test batches  : 24
